# 04 — Service & Travel Proxy Models

**Purpose**: Demonstrate the service-model components built in Phase 2:

1. **Distance matrix** — Haversine distances between 48 Manhattan firehouses and 30 precinct centroids.
2. **Travel-time proxy** — Distance-based travel times with optional time-of-day speed variation.
3. **Service-time distribution** — LogNormal on-scene service times.
4. **NHPP Arrival Generator** — Non-homogeneous Poisson arrivals from lambda tables.

All modules live in `src/ems_readiness/`.


### Auto-generate missing data
The cell below checks if required processed data exists and generates it automatically if missing.
This ensures each notebook can run independently from a clean state.

In [ ]:
# Auto-generate missing processed data if needed
import sys, os
from pathlib import Path

# Detect project root (works from notebooks/ directory)
_PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
if not (_PROJECT_ROOT / 'scripts' / 'generate_all_data.py').exists():
    _PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(_PROJECT_ROOT))
sys.path.insert(0, str(_PROJECT_ROOT / 'src'))

from scripts.generate_all_data import ensure_data
ensure_data(_PROJECT_ROOT)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.pardir, 'src'))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

from ems_readiness.utils.distance import haversine, build_distance_matrix
from ems_readiness.service.travel_time import (
    travel_time_minutes, build_travel_time_matrix, TOD_SPEED_FACTORS, DEFAULT_SPEED_MPH,
)
from ems_readiness.service.service_time import ServiceTimeModel
from ems_readiness.demand.arrival_generator import NHPPArrivalGenerator

print("All modules imported successfully [PASS]")


## 1. Distance Matrix (Firehouse -> Precinct Centroid)

In [ ]:
# Load pre-computed distance matrix
dm = pd.read_csv('../data/processed/distance_matrix_firehouse_precinct.csv', index_col=0)
dm.columns = dm.columns.astype(str)
print(f"Shape: {dm.shape}  (firehouses × precincts)")
print(f"Range: {dm.min().min():.3f} – {dm.max().max():.3f} miles")
print(f"Mean : {dm.values.mean():.3f} miles")
dm.head()


In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(dm, cmap='YlOrRd', annot=False, fmt='.1f', ax=ax,
            cbar_kws={'label': 'Haversine Distance (miles)'})
ax.set_title('Distance Matrix: Firehouses -> Precinct Centroids')
ax.set_xlabel('Precinct')
ax.set_ylabel('Firehouse')
plt.tight_layout()
plt.savefig('../results/baseline/figures/distance_matrix_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Nearest firehouse for each precinct
nearest = dm.idxmin(axis=0)
nearest_dist = dm.min(axis=0)

summary = pd.DataFrame({
    'Precinct': nearest.index,
    'Nearest Firehouse': nearest.values,
    'Distance (mi)': nearest_dist.values,
})
summary = summary.sort_values('Distance (mi)')
print("Nearest firehouse for each precinct:")
summary.head(10)


## 2. Travel-Time Proxy

In [ ]:
# Base travel time matrix (no TOD adjustment)
tt_base = build_travel_time_matrix(dm, speed_mph=20.0)
print(f"Travel time range: {tt_base.min().min():.1f} – {tt_base.max().max():.1f} minutes")
print(f"Mean travel time: {tt_base.values.mean():.1f} minutes")


In [ ]:
# Compare travel times across different times of day
hours_to_compare = [2, 8, 12, 18, 22]
results = {}
for h in hours_to_compare:
    tt_h = build_travel_time_matrix(dm, speed_mph=20.0, hour_of_day=h)
    results[f"Hour {h:02d}"] = tt_h.values.flatten()

fig, ax = plt.subplots(figsize=(10, 6))
pd.DataFrame(results).plot.box(ax=ax)
ax.set_ylabel('Travel Time (minutes)')
ax.set_title('Travel Time Distribution by Time of Day')
plt.tight_layout()
plt.savefig('../results/baseline/figures/travel_time_by_tod.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# TOD speed factors visualization
hours = list(range(24))
factors = [TOD_SPEED_FACTORS[h] for h in hours]
speeds = [DEFAULT_SPEED_MPH * f for f in factors]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.bar(hours, factors, color='steelblue', alpha=0.8)
ax1.axhline(1.0, color='red', linestyle='--', alpha=0.5)
ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('Speed Factor')
ax1.set_title('Time-of-Day Speed Multipliers')

ax2.bar(hours, speeds, color='coral', alpha=0.8)
ax2.axhline(DEFAULT_SPEED_MPH, color='red', linestyle='--', alpha=0.5, label=f'Base: {DEFAULT_SPEED_MPH} mph')
ax2.set_xlabel('Hour of Day')
ax2.set_ylabel('Effective Speed (mph)')
ax2.set_title('Effective EMS Speed by Hour')
ax2.legend()

plt.tight_layout()
plt.savefig('../results/baseline/figures/tod_speed_factors.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Service-Time Distribution

In [ ]:
# Default model: LogNormal(mean=25, std=10)
model = ServiceTimeModel(mean_minutes=25.0, std_minutes=10.0, distribution='lognormal')
print(model)

samples = model.sample(10_000, rng=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(samples, bins=60, density=True, alpha=0.7, color='steelblue', edgecolor='white')
axes[0].axvline(np.mean(samples), color='red', linestyle='--', label=f'Mean = {np.mean(samples):.1f} min')
axes[0].axvline(np.median(samples), color='orange', linestyle='--', label=f'Median = {np.median(samples):.1f} min')
axes[0].set_xlabel('Service Time (minutes)')
axes[0].set_ylabel('Density')
axes[0].set_title('LogNormal Service Time Distribution')
axes[0].legend()
axes[0].set_xlim(0, 80)

# Compare with Exponential
model_exp = ServiceTimeModel(mean_minutes=25.0, distribution='exponential')
samples_exp = model_exp.sample(10_000, rng=42)
axes[1].hist(samples, bins=60, density=True, alpha=0.5, label='LogNormal', color='steelblue')
axes[1].hist(samples_exp, bins=60, density=True, alpha=0.5, label='Exponential', color='coral')
axes[1].set_xlabel('Service Time (minutes)')
axes[1].set_ylabel('Density')
axes[1].set_title('LogNormal vs Exponential')
axes[1].legend()
axes[1].set_xlim(0, 100)

plt.tight_layout()
plt.savefig('../results/baseline/figures/service_time_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nLogNormal  — Mean: {np.mean(samples):.1f}, Median: {np.median(samples):.1f}, Std: {np.std(samples):.1f}")
print(f"Exponential — Mean: {np.mean(samples_exp):.1f}, Median: {np.median(samples_exp):.1f}, Std: {np.std(samples_exp):.1f}")


## 4. NHPP Arrival Generator Demo

In [ ]:
# Load from pre-computed lambda tables
gen = NHPPArrivalGenerator.from_tables('../data/processed', base_rate=3.48)
print(gen)

# Generate 24h of arrivals (Monday)
arrivals = gen.generate_arrivals(n_hours=24, start_hour=0, dow=0, rng=42)
print(f"\nGenerated {len(arrivals)} arrivals in 24 hours")
print(f"Expected ≈ {3.48 * 24 * gen.dow_factors[0]:.0f} arrivals")
arrivals.head(10)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hourly arrival counts
hourly_counts = arrivals.groupby('hour').size()
axes[0].bar(hourly_counts.index, hourly_counts.values, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Number of Arrivals')
axes[0].set_title('Simulated Arrivals by Hour (Single Replication)')

# Precinct distribution
if 'precinct' in arrivals.columns:
    pct_counts = arrivals.groupby('precinct').size().sort_values(ascending=False)
    axes[1].barh(pct_counts.index.astype(str), pct_counts.values, color='coral', alpha=0.8)
    axes[1].set_xlabel('Number of Arrivals')
    axes[1].set_ylabel('Precinct')
    axes[1].set_title('Simulated Arrivals by Precinct')

plt.tight_layout()
plt.savefig('../results/baseline/figures/nhpp_arrivals_demo.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Integration: Full Call Timeline Example

In [ ]:
# Simulate a single EMS call
rng = np.random.default_rng(42)
service_model = ServiceTimeModel(mean_minutes=25.0, std_minutes=10.0)
dispatch_delay = 1.5  # minutes

# Pick a random arrival
call = arrivals.iloc[0]
call_hour = int(call['hour'])
call_precinct = int(call['precinct']) if 'precinct' in call.index else 1

# Find nearest firehouse for that precinct
precinct_col = str(call_precinct)
if precinct_col in dm.columns:
    nearest_fh = dm[precinct_col].idxmin()
    distance = dm.loc[nearest_fh, precinct_col]
else:
    nearest_fh = dm.iloc[:, 0].idxmin()
    distance = dm.iloc[:, 0].min()

travel = travel_time_minutes(distance, speed_mph=20.0, hour_of_day=call_hour)
on_scene = service_model.sample(1, rng=rng)[0]

print("" * 55)
print("  EXAMPLE EMS CALL TIMELINE")
print("" * 55)
print(f"  Call hour:        {call_hour:02d}:00")
print(f"  Precinct:         {call_precinct}")
print(f"  Nearest firehouse: {nearest_fh}")
print(f"  Distance:         {distance:.2f} miles")
print(f"  ")
print(f"  1. Dispatch delay:  {dispatch_delay:.1f} min")
print(f"  2. Travel to scene: {travel:.1f} min")
print(f"  3. On-scene service: {on_scene:.1f} min")
print(f"  ")
total = dispatch_delay + travel + on_scene
print(f"  TOTAL:              {total:.1f} min")
print("" * 55)


## Summary

| Component | Implementation | Key Parameters |
|-----------|---------------|----------------|
| **Distance** | Haversine (great-circle) | 48 firehouses × 30 precincts |
| **Travel time** | distance / speed | 20 mph base, TOD factors |
| **Service time** | LogNormal | μ=25 min, σ=10 min |
| **Arrivals** | NHPP (thinning) | λ₀=3.48/hr, hourly+DOW factors |

All parameters are configurable in `configs/service.yaml` and `configs/demand.yaml`.

**Next step** -> Phase 3: SimPy discrete-event simulation integrating these components.
